# **Win 区布局**

基于前一节对 Dispatch 与 Combine 算法层“写入 Win 区→在 Win 区完成同步→从 Win 区读出”数据流的分析，本节进一步说明 Win 区的划分方式、写入位置和共享机制。

**学习目标**：掌握 Win 区中「Win数据区 / Win状态区」「ping-pong 翻转」「Dispatch / Combine 子区」三层划分；理解 token 在 Win 上为何切成 480 + 32 字节的小块。

**关键常量速查表**（来自实践代码 [moe_distribute_comm.h](./answer/scaffold_02_10/include/moe_distribute_comm.h) / [moe_distribute_dispatch_non_quant.h](./answer/scaffold_02_10/include/moe_distribute_dispatch_non_quant.h) / [moe_distribute_combine.h](./answer/scaffold_02_10/include/moe_distribute_combine.h)）：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">常量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">取值</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">用途</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>STATUS_REGION_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>1022 × 1024 × 1024</code> 字节</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">同一 Win 区中「Win状态区」相对「Win数据区」的固定偏移</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>BUFFER_NUM</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>2</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win 区按 ping-pong 切两半，相邻两次调用交替使用</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>SPLIT_BLOCK_DATA_SIZE</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>480 B</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每个 token 在 Win数据区上的「数据子块」大小</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>SPLIT_BLOCK_SIZE</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>512 B</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">数据子块 + 紧随其后的 32 B token-flag 的合计大小（512 = 480 + 32）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>WIN_ADDR_ALIGN</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>512 B</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win数据区上的地址对齐粒度</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>UB_ALIGN</code> / <code>STATE_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>32 B</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">UB 与 Win状态区最小对齐单元（8 个 int32 元素）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Dispatch <code>WIN_STATE_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>384 × 1024</code> 字节</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Dispatch 视角的 Win状态区 ping-pong 步长</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Combine <code>WIN_STATE_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>500 × 1024</code> 字节</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Combine 视角的 Win状态区 ping-pong 步长</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>COMBINE_STATE_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>64 × 1024</code> 字节</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Combine 状态区在 dispatch 状态区之后预留的起始偏移</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>FLAG_FIELD_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>768 × 1024</code> 字节</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win状态区上记录「当前 ping-pong half」的 0/1 标识区位置</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>CUMSUM_CAL_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>868 × 1024</code> 字节</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win状态区上核间软同步的汇总区位置</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>CUMSUM_FLAG_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>876 × 1024</code> 字节</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win状态区上「本地整理同步 flag」位置</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>STATE_WIN_OFFSET</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>975 × 1024</code> 字节</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Combine 中预留 <code>48 × 512</code> 字节状态空间的起始偏移</td>
  </tr>
</table>

---

## **1. 两个视图：数据 vs 状态**

Win 区对每张卡而言都是同一段连续物理内存。代码里以两条索引函数把这段内存「分视图」：

- `GetShmemDataAddr(pe)`：返回对端 `pe` 上的数据基址，等于 `aclshmem_ptr(pe)`。
- `GetShmemSignalAddr(pe)`：返回对端 `pe` 上的状态基址，等于 `aclshmem_ptr(pe) + STATUS_REGION_OFFSET`，其中 `STATUS_REGION_OFFSET = 1022 MiB`。

因此 **同一块 1024 MiB 量级的 Win 区被划成两段**：前 1022 MiB 用作「Win数据区」，后 2 MiB 用作「Win状态区」。Dispatch / Combine 写到「对端 Win数据区」时，地址都是从 `GetShmemDataAddr(pe)` 起步加上具体偏移；写「对端 Win状态区」时，地址都是从 `GetShmemSignalAddr(pe)` 起步加上具体偏移。

<img src="./images/02.07_data_area_and_status_area.png" width="800">

**实现说明**：「Win数据区」与「Win状态区」物理上是同一块内存的不同段，并非两块独立的 HBM；之所以这样切分，是为了让数据写入（大块 DMA）与状态写入（少量 flag/count）走不同的偏移基址，便于代码与硬件队列独立调度。

---

## **2. Ping-pong 翻转：每次调用换半区**

Dispatch / Combine 算子在训练中会被反复调用。为避免「上一次的状态/数据还没被对端读完，本次又开始覆盖」，Win 区在Win数据区与Win状态区上都按 `BUFFER_NUM = 2` 切成 **ping-pong 两半**，本次用哪一半由一个 0/1 标识 `dataState_` 决定。

**Ping-pong 标识本身存在 Win状态区上**：`FLAG_FIELD_OFFSET = 768 KB` 位置专门留给这个标识，每个 AIV 核占一个 `WIN_ADDR_ALIGN = 512 B` 槽位。每次 `Init` 时由 `InitWinState` 读出该槽位的旧值作为 `dataState_`，并把翻转值写回供下次使用。

**Half buffer 步长**（Dispatch / Combine 各有一套，因为同一对算子复用同一块 Win，但二者状态区子分区方式不一样）：

- Dispatch 视角：Win状态区按 `WIN_STATE_OFFSET = 384 KB` 翻转，Win数据区按 `totalWinSize_ / 2` 翻转；
- Combine 视角：Win状态区按 `WIN_STATE_OFFSET = 500 KB` 翻转（与 dispatch 错开），Win数据区按 `totalWinSize_ / 2` 翻转。

Dispatch / Combine 都在 `Init` 阶段把翻转后的偏移记进各自的 `winDataSizeOffset_` 与 `winStatusOffset_`，后续所有 `GetWindAddrByRankId` / `GetWinStateAddrByRankId` 调用都已经包含 ping-pong 偏移，业务代码不再感知 ping-pong 本身。

![整理前后布局](./images/02.07_ping-pong_flip.png)

**实现说明**：`dataState_` 不是「本次调用产生的标记」，而是「Init 阶段从对端读到的、上次调用留下的」。每个核独立读自己槽位，但读到的应当一致（写入与翻转都由所有核遵循同一规则）。

---

## **3. Win数据区子结构：Combine 预留段 + Dispatch 数据段**

Win数据区的每一个 ping-pong half 都被 Dispatch 与 Combine 共用，二者切法如下（图 G）：

- **Combine 预留段**：位于 half 起点 `0 ~ axisBS × axisK × hSizeAlignCombine` 字节，其中 `hSizeAlignCombine = ceil(h * sizeof(X), 512) × 512`，即按 512 B 对齐每条 token 的 K 个 topk 槽。[Combine 2.1 阶段回送](./01.06_dispatch_combine_core_flow.ipynb)的 token 就写到这一段：第 i 条本地 token 的第 k 个 topk 槽位于 `(i × K + k) × hSizeAlignCombine`。
- **Dispatch 数据段**：紧接在 Combine 预留段之后，从 `axisBS × axisK × hSizeAlignCombine` 一直占到 half 末尾。[Dispatch 1.1阶段写到对端](./01.06_dispatch_combine_core_flow.ipynb)阶段写到对端的 token（按 (本地专家, 源卡, token 序号) 排列）落在这里。

代码对应（`SetDataStatus`）：

- Dispatch：`winDataSizeOffset_ = dataState_ × (totalWinSize_ / 2) + axisBS × axisK × hSizeAlignCombine` —— 跳过 Combine 预留段。
- Combine：`winDataSizeOffset_ = dataState_ × (totalWinSize_ / 2)` —— 直接落在 half 起点的 Combine 预留段上。

<img src="./images/02.07_tokens_arranged_in_the_win_data_area.png" width="800">

**实现说明**：

- Combine 预留段与 Dispatch 数据段处于同一 ping-pong half；本轮 Dispatch 写在 half X，本轮 Combine 也写在 half X（不同区段），下一轮整体翻到 half ¬X。
- 「`hSizeAlignCombine = 512 对齐`」与 Dispatch 数据段中 token 的「480 + 32 切分」是两个独立约束：前者保证 Combine 槽位地址 512 对齐；后者用于 Dispatch 把 token-flag 嵌入到数据流中（见下一节）。

---

## **4. Dispatch 数据段中 token 的切分：480 + 32**

Dispatch 把一条 token 写到对端时，并不是「连续 `h × sizeof(X)` 字节一气呵成」，而是按 **480 B 数据子块 + 32 B token-flag** 交替排布。这一切分由两个常量决定：

- `SPLIT_BLOCK_DATA_SIZE = 480 B`：每个数据子块的有效数据长度；
- `SPLIT_BLOCK_SIZE = 512 B`：每个数据子块 + 紧随其后 32 B token-flag 的合计长度。

对应代码（`SetTilingDataAndCal`）：

- `hOutSizeAlign_ = ceil(h × sizeof(X), 32) × 32 + 32 B`（先按 UB 对齐，再额外 +32 B 给末尾的 token-flag 留位）；
- `blockCntPerToken_ = ceil(hOutSizeAlign_, 480)`（一条 token 切成多少个 480 B 子块）；
- `hCommuSize_ = blockCntPerToken_ × 512 B`（一条 token 在 Win 上的完整占位 = 子块数 × 512）。

**每个 32 B token-flag 槽位的语义**：当 Dispatch 把一个数据子块成功推送到对端 Win 后，会紧跟着把一个 UB 对齐块（首位 `1.0`，其余 0）写到该子块尾部的 32 B 区域。[Dispatch 1.4 阶段](./01.06_dispatch_combine_core_flow.ipynb)在搬运对端 token 之前，先 `Sum` 这些 token-flag——求和等于「期望的子块数」即代表该 token 全部到齐。这就是 1.4 中提到的「token 粒度 flag」的物理形式。

**图 H · 一条 token 在 Win数据区上的物理排布**：

![一条token在Win数据区上的物理排布](./images/02.07_tokens_arranged_in_the_Win_data_area_and_status_area.png)

**为什么这样切**：

- 把 token-flag 嵌入到数据流中、而非单独成一张表，可以让对端用一次连续读取同时取得「数据 + 到达标记」，不需要额外的随机访问。
- 480 B 与 32 B 合成 512 B，恰好对齐 `WIN_ADDR_ALIGN = 512`，与共享内存写指令的最佳粒度吻合。
- 切多个小块（每块独立 flag）而不是「整条 token 一个 flag」，是为了允许「部分到达就开始整理」的细粒度判断；1.4 阶段对每条 token 把 `blockCntPerToken_` 个 flag 求和后再决定是否搬运。

**实现说明**：Combine 预留段中每条 token 的占位 `hSizeAlignCombine`（仅 512 对齐，不切 480+32）与 Dispatch 数据段中 token 的占位 `hCommuSize_`（多个 480+32 块）是两套不同的布局。前者只用 K 路 token-flag（每条 32 B）单独放在状态区上，后者把 token-flag 嵌入数据流。

---

## **5. Win状态区子结构**

Win状态区同样按 ping-pong 切两半，每一半内部又划分多个固定偏移子区（Dispatch / Combine 各按自己的偏移常量使用）。

**Dispatch 视角**（一个 384 KB half 内）：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">偏移</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">大小</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">用途</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>0</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">数 KB</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">状态块阵列（Dispatch 1.2 写入：每个 (源卡, 本地专家) 对应一个 32 B 状态块，首位 flag，第二位 count）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>COMBINE_STATE_OFFSET = 64 KB</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">数 KB</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">让出给 Combine 状态使用（Combine 在此基础上再加 ping-pong 偏移）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>CUMSUM_CAL_OFFSET = 868 KB</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">8 B</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">跨 ping-pong half 的核间汇总区（Dispatch 1.3 各核写自己汇总值给同组核轮询）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>CUMSUM_FLAG_OFFSET = 876 KB</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">数 B</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本地整理同步 flag（Dispatch 1.3 末尾置位，通知 1.4 偏移已就绪）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>FLAG_FIELD_OFFSET = 768 KB</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aivNum × 512 B</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">0/1 ping-pong 标识区，每核一槽</td>
  </tr>
</table>

**Combine 视角**：每个 half 在 Dispatch 的状态空间之后偏移 `COMBINE_STATE_OFFSET = 64 KB` 开始，按 `WIN_STATE_OFFSET = 500 KB` 翻转。

- `winStatusOffset_ = COMBINE_STATE_OFFSET + dataState_ × WIN_STATE_OFFSET`。
- 内部 token-flag 阵列：第 `i` 条本地 token 的第 `k` 个 topk-flag 位于 `i × axisK × STATE_OFFSET + k × STATE_OFFSET`，其中 `STATE_OFFSET = 32 B`。[Combine 2.1](./02.06_dispatch_combine_core_flow.ipynb) 写到对端、[Combine 2.2](./02.06_dispatch_combine_core_flow.ipynb) 在本卡 `Sum` 求和。
- `STATE_WIN_OFFSET = 975 KB`：Combine 还会再预留 `48 × 512 = 24 KB` 的额外状态空间，用于 reset 等辅助操作。

![一条token在Win数据区上的物理排布](./images/02.07_tokens_arranged_in_the_win_status_area.png)

**实现说明**：

- Dispatch 与 Combine 的 `WIN_STATE_OFFSET` 不同（384 KB vs 500 KB），意味着二者的 ping-pong half 在物理上 **并不完全重合**；这是因为 Dispatch 状态分区在前、Combine 状态分区在后，二者各自按自己的步长翻转，互不干扰。
- `FLAG_FIELD_OFFSET / CUMSUM_*` 这几个常量都属于 Dispatch namespace，是 Dispatch 流程独占的；Combine 没有「核间软同步 + 前缀和」这一阶段，因此不需要类似槽位。

---

## **6. 关键设计点小结**

- **物理一块，视图两套**：Win 区以 `STATUS_REGION_OFFSET = 1022 MiB` 为界，把同一段物理内存分成「Win数据区 / Win状态区」，业务代码用两条地址工具函数访问，避免数据写入与状态写入相互踩到。
- **Ping-pong 用一比特状态承载**：`dataState_` 的 0/1 完整决定了一次调用使用哪一半 Win 区，标识本身存在 Win状态区上、由 `Init` 阶段读取并翻转，业务代码完全感知不到 ping-pong 切换。
- **数据区两段共用一个 half**：Combine 预留段位于 half 起点（按 K 路、512 对齐布局），Dispatch 数据段紧随其后（按 480 + 32 切分布局）；两段共用同一 ping-pong half，但写入互不重叠。
- **token-flag 与数据同流**：Dispatch 段中 480 B 数据后紧跟 32 B token-flag，使对端可以一次连续 DMA 拿到「数据 + 到达标记」，并支持按子块粒度判断到齐；Combine 段则采用「数据写数据区、flag 写状态区」的分离模式，因 Combine 的 token-flag 单条 32 B 即够用。
- **状态区子区固定偏移**：所有同步槽位（ping-pong 标识、cumsum 汇总区、cumsum flag、Combine 状态块）位置都由编译期常量决定，无需运行时协商，符合 "位置即协议" 的整体设计哲学。

---

**导航**：[← 上一章：核心流程拆解](02.06_dispatch_combine_core_flow.ipynb) | [返回课程主页](02.01_chapter_intro.ipynb) | [下一章：Tiling开发指引 →](02.08_tiling_guide.ipynb)

---